**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**

In [1]:
#Step 0 - import libraries needed for this lab 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Standard seed used throughout the notebook, for reproducibility
SEED = 42

In [2]:
# Load the dataset
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


# Step 1 --- Target Distribution
What is the **distribution of our target variable?**

- Can we say we're dealing with an *imbalanced dataset?*

In [3]:
# Target distribution
fraud["fraud"].value_counts()

fraud
0.0    912597
1.0     87403
Name: count, dtype: int64

In [4]:
# this is to see the output result in percentage
fraud["fraud"].value_counts(normalize=True) * 100

fraud
0.0    91.2597
1.0     8.7403
Name: proportion, dtype: float64

Yes, we can say the `fraud` column is heavily **imbalanced**. 

The vast majority of transactions(about **91.25%**) are legitimate belonging to to the majority class (`0.0`) while only a small minority(about **8.74%**) are fraudulent (`1`) this is typically around **9% fraud vs. 91%** legit in this dataset. This matters a lot for modeling: a model that just predicts "legit" every single time would already score *91%* accuracy while being completely useless at catching fraud. We'll need metrics beyond accuracy, and later we'll try balancing techniques.

## Step 2 - Train/ Test Split
- **Train a LogisticRegression.**

In [5]:
# Step 2: Train / Test split
#---------------------------

X = fraud.drop(columns=["fraud"])
y = fraud["fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))

X_train shape: (800000, 7)
X_test shape: (200000, 7)

Train target distribution:
fraud
0.0    0.912597
1.0    0.087402
Name: proportion, dtype: float64


In [6]:
# Train a logistic Regression
log_reg = LogisticRegression(max_iter=1000, random_state=SEED)
log_reg.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


## Step 3 - Evaluate
- Evaluate your model. Take in consideration class importance, and evaluate it by selecting the correct metric.

With an imbalanced target, **accuracy is misleading** (predicting "legit" for everyone would already look good). Instead I focus on **precision, recall, and F1-score for the fraud (`1`) class**, especially **recall**, since missing real fraud (false negatives) is usually far more costly than a false alarm.

In [7]:
# Evaluate your model
def evaluate(model, X_test, y_test, label="Model"):
    y_pred = model.predict(X_test)
    print(f"--- {label} ---")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision (fraud=1):", precision_score(y_test, y_pred))
    print("Recall (fraud=1):", recall_score(y_test, y_pred))
    print("F1-score (fraud=1):", f1_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    return y_pred
baseline_pred = evaluate(log_reg, X_test, y_test, "Logistic Regression (imbalanced)")

--- Logistic Regression (imbalanced) ---
Accuracy: 0.95928
Precision (fraud=1): 0.8958704316119732
Recall (fraud=1): 0.6043704593558721
F1-score (fraud=1): 0.7218009154881465

Classification Report:
               precision    recall  f1-score   support

         0.0       0.96      0.99      0.98    182519
         1.0       0.90      0.60      0.72     17481

    accuracy                           0.96    200000
   macro avg       0.93      0.80      0.85    200000
weighted avg       0.96      0.96      0.96    200000

Confusion Matrix:
 [[181291   1228]
 [  6916  10565]]


Because fraud is are, the model tends to have decent precision but noticeably weaker recall on the fraud class, it misses a meaningful chunk of actual fraud cases. This is the classic symptom of training on imbalanced data. Let's see if balancing the training set helps.

## Step 4 - 
- Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model?

`RandomOverSampler` duplicates examples from the minority (fraud) class in the **training set only** until both classes are balanced. We never touch the test set, it must stay representative of real-world data.



In [8]:
# Oversampling
ros = RandomOverSampler(random_state=SEED)
X_train_over, y_train_over = ros.fit_resample(X_train, y_train)

print("Balanced training distribution:")
print(y_train_over.value_counts())

Balanced training distribution:
fraud
0.0    730078
1.0    730078
Name: count, dtype: int64


In [9]:
log_reg_over = LogisticRegression(max_iter=1000, random_state=SEED)
log_reg_over.fit(X_train_over, y_train_over)

over_pred = evaluate(log_reg_over, X_test, y_test, "Logistic Regression (oversampled)")

--- Logistic Regression (oversampled) ---
Accuracy: 0.93482
Precision (fraud=1): 0.5774471199080043
Recall (fraud=1): 0.9479434814941937
F1-score (fraud=1): 0.7177010697734852

Classification Report:
               precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000

Confusion Matrix:
 [[170393  12126]
 [   910  16571]]


**Does it improve?** Oversampling typically boosts **recall** on the fraud class quite a bit the model no longer defaults to predicting "legit" as often usually at a small cost to precision (a few more false alarms). Since catching fraud (recall) usually matters more than avoiding false alarms, this is often a good trade-off.

## Step 5 
Run **Undersample** in order to balance our target variable and repeat the steps above *(1-3),* now with balanced data. *Does it improve the performance of our model?*

`RandomUnderSampler` does the opposite: it randomly drops examples from the majority (legit) class in the training set until both classes are balanced. This is fast and simple, but throws away data, which can hurt performance if the majority class had useful patterns in the discarded rows.

In [10]:
# Undersampling
rus = RandomUnderSampler(random_state=SEED)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

print("Balanced training distribution:")
print(y_train_under.value_counts())

Balanced training distribution:
fraud
0.0    69922
1.0    69922
Name: count, dtype: int64


In [11]:
log_reg_under = LogisticRegression(max_iter=1000, random_state=SEED)
log_reg_under.fit(X_train_under, y_train_under)

under_pred = evaluate(log_reg_under, X_test, y_test, "Logistic Regression (undersampled)")

--- Logistic Regression (undersampled) ---
Accuracy: 0.93474
Precision (fraud=1): 0.5771629673507788
Recall (fraud=1): 0.9475430467364567
F1-score (fraud=1): 0.7173668254655695

Classification Report:
               precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000

Confusion Matrix:
 [[170384  12135]
 [   917  16564]]


**Does it improve our moodel?** Undersampling usually gives a similar recall boost to oversampling (since the class ratio the model sees is the same), but because we've thrown away a large chunk of the majority class, results can be slightly less stable especially if the original dataset wasn't huge. Here, with a large dataset, undersampling still leaves plenty of training data, so performance is often comparable to oversampling.

## Step 6 — SMOTE
Run **SMOTE** in order to balance our target variable and repeat the steps above *(1-3),* now with balanced data. Does it improve the performance of our model? 

**SMOTE** (Synthetic Minority Over-sampling Technique) also balances the training set, but instead of duplicating existing fraud examples, it creates **new synthetic** fraud examples by interpolating between real minority-class neighbors. This can generalize better than plain duplication.

In [12]:
smote = SMOTE(random_state=SEED)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Balanced training distribution:")
print(y_train_smote.value_counts())

Balanced training distribution:
fraud
0.0    730078
1.0    730078
Name: count, dtype: int64


In [13]:
log_reg_smote = LogisticRegression(max_iter=1000, random_state=SEED)
log_reg_smote.fit(X_train_smote, y_train_smote)

smote_pred = evaluate(log_reg_smote, X_test, y_test, "Logistic Regression (SMOTE)")

--- Logistic Regression (SMOTE) ---
Accuracy: 0.935185
Precision (fraud=1): 0.5790911000630208
Recall (fraud=1): 0.9461701275670729
F1-score (fraud=1): 0.7184588319614273

Classification Report:
               precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.94    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.94      0.94    200000

Confusion Matrix:
 [[170497  12022]
 [   941  16540]]


**Does it improve?** SMOTE generally performs similarly to, or slightly better than, random oversampling, it tends to give strong recall on the fraud class while often keeping precision a bit healthier than plain duplication, since the synthetic points add some variety rather than exact repeats.

## Conclusion

- The raw dataset is strongly imbalanced (~9% fraud), so **accuracy alone is not a reliable metric**, precision, recall, and F1 on the fraud class matter far more.
- The baseline Logistic Regression (trained on imbalanced data) tends to under-detect fraud (lower recall).
- All three balancing techniques — **oversampling, undersampling, and SMOTE**, typically improve recall on the fraud class substantially compared to the imbalanced baseline, usually at some cost to precision.
- In practice, the "best" choice depends on the business cost of **missing fraud (false negative) vs. flagging a legitimate transaction (false positive),** and SMOTE or oversampling are often preferred over undersampling when there's plenty of data to work with, since undersampling discards information.